In [ ]:
# 📦 Install LangChain v1 and the integration packages used in this notebook
# Pinned to the version requested: langchain==1.3.11
#%pip install -q "langchain==1.3.11" langchain-openai langchain-tavily langgraph python-dotenv

As of **LangChain v1**, `initialize_agent` and `AgentType` are deprecated and have moved to the `langchain-classic` package. The new standard for building agents is [`create_agent`](https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent) — a single, unified, middleware-driven agent constructor built on LangGraph, replacing `langgraph.prebuilt.create_react_agent` as well as every `AgentType` variant.

Each section below reproduces the *intent* of one of the original 7 agent types using `create_agent`, plus a bonus section on **middleware** — the core new capability in v1 that replaces most of what the old `AgentType` variants used to hard-code.

Reference: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>


## Migration cheat-sheet: `AgentType` → LangChain v1

| # | Legacy `AgentType` (pre-v1) | What it did | v1 equivalent with `create_agent` |
|---|---|---|---|
| 1 | `ZERO_SHOT_REACT_DESCRIPTION` | Single-shot ReAct loop, no memory | `create_agent(model, tools)` — the default agent loop |
| 2 | `CONVERSATIONAL_REACT_DESCRIPTION` | ReAct + `ConversationBufferMemory` | `create_agent(..., checkpointer=InMemorySaver())` + `thread_id` |
| 3 | `CHAT_CONVERSATIONAL_REACT_DESCRIPTION` | Chat-model ReAct + memory | Same as above — chat models are the default everywhere now |
| 4 | `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION` | Multi-argument tools via a structured schema | Any `@tool`-decorated function passed straight to `create_agent` |
| 5 | `OPENAI_FUNCTIONS` | Function-calling instead of text-based ReAct parsing | Native tool calling — `create_agent` always uses the model's function/tool-calling API when available |
| 6 | `OPENAI_MULTI_FUNCTIONS` | Multiple function calls in a single step | Native parallel tool calling — no special agent type needed |
| 7 | `SELF_ASK_WITH_SEARCH` | Explicit `Follow up: ... / Intermediate answer: ...` prompting pattern | A `system_prompt` that encodes the self-ask pattern, combined with a search tool |

Other things that moved or were renamed:

- `langchain.chat_models.ChatOpenAI` → `langchain_openai.ChatOpenAI` (or `langchain.chat_models.init_chat_model`)
- `langchain.chains.LLMChain` → moved to `langchain-classic`; prefer a plain `@tool` function or an LCEL chain
- `langchain.memory.ConversationBufferMemory` → replaced by LangGraph **checkpointers** (`InMemorySaver`, `PostgresSaver`, ...)
- `langchain.tools.TavilySearchResults` (community, deprecated) → `langchain_tavily.TavilySearch`


In [1]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# 🔐 Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# NOTE: gpt-3.5-turbo (used in the original notebook) has been retired.
# Swap in whichever current chat model you have access to.
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# --- Tools -------------------------------------------------------------

@tool
def simple_qa(question: str) -> str:
    """Answer a question clearly and directly, without hallucinating."""
    response = model.invoke(f"Answer clearly: {question}")
    return response.content

# Web search tool (Tavily) — replaces the deprecated TavilySearchResults
web_search = TavilySearch(max_results=3)

tools = [simple_qa, web_search]

## 1) Basic agent (replaces `ZERO_SHOT_REACT_DESCRIPTION`)

The zero-shot ReAct agent was the "simple + reliable default" in the legacy API — no memory, single-pass reasoning + tool use. In v1 this is simply `create_agent` with no `checkpointer` and no extra middleware.

In [2]:
#)1.Basic agent (replaces ZERO_SHOT_REACT_DESCRIPTION)
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when they help you answer accurately.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]}
)

print(result["messages"][-1].content)

LangChain is a framework that helps developers build applications using large language models (LLMs) more effectively by managing prompts, chaining model calls, and integrating external data sources. It was created by Harrison Chase.


## 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)

`ConversationBufferMemory` is gone. Short-term memory in v1 is handled by a **checkpointer** attached to the agent, addressed by a `thread_id`. Every `invoke()` call on the same thread automatically sees the full prior message history — no manual memory object to wire up.

In [3]:
# 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
)

thread_config = {"configurable": {"thread_id": "agent-types-demo-1"}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
    thread_config,
)
print(result["messages"][-1].content)

LangChain is a framework designed to simplify building applications with large language models by enabling easy integration of various data sources and tools. It facilitates creating complex workflows that combine LLMs with external APIs, databases, and custom logic. LangChain was created by Harrison Chase.


## 3) Multi-turn chat agent (replaces `CHAT_CONVERSATIONAL_REACT_DESCRIPTION`)

`CHAT_CONVERSATIONAL_REACT_DESCRIPTION` mainly existed to point the conversational agent at a chat model instead of a plain text-completion LLM. In v1 every model is a chat model, so that distinction disappears — the memory-backed agent above already gives us multi-turn chat. Here we simply continue the **same thread** to show a follow-up question that depends on prior context:

In [4]:
## 3) Multi-turn chat agent (replaces `CHAT_CONVERSATIONAL_REACT_DESCRIPTION`)
follow_up = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Now plan a 3-step study path for what I just asked about."}]},
    thread_config,  # same thread_id -> agent remembers the prior turn
)
print(follow_up["messages"][-1].content)

Here is a 3-step study path to learn about LangChain and its creator:

Step 1: Understand the Basics of LangChain
- Study the core concepts of LangChain, including its purpose, how it integrates large language models with external data sources, and the types of applications it enables.
- Review introductory tutorials or official documentation to get a foundational understanding.

Step 2: Explore LangChain's Features and Use Cases
- Dive deeper into LangChain's capabilities such as connecting to APIs, databases, and creating complex workflows.
- Look at example projects or case studies to see how LangChain is applied in real-world scenarios.

Step 3: Learn About Harrison Chase and the Development of LangChain
- Research Harrison Chase, the creator of LangChain, to understand his background and motivations for developing the framework.
- Follow his talks, blog posts, or interviews to gain insights into the vision and future direction of LangChain.

This path will give you a comprehensive

## 4) Structured, multi-argument tools (replaces `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION`)

`StructuredTool` + a Pydantic `args_schema` used to require its own dedicated `AgentType`. In v1, **every** tool passed to `create_agent` supports multi-argument, typed, structured calling out of the box — no special agent type required. The `@tool` decorator infers the schema straight from your function's type hints and docstring.

In [5]:
## 4) Structured, multi-argument tools (replaces `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION`)
@tool
def title_maker(topic: str, tone: str = "concise") -> str:
    """Generate a title given a topic and an optional tone (e.g. 'concise', 'playful', 'formal')."""
    return f"{tone.title()} Title: {topic} in Practice"

structured_agent = create_agent(
    model=model,
    tools=[title_maker],
)

result = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "Make a friendly title about LangGraph tutorials."}]}
)
print(result["messages"][-1].content)

A friendly title for LangGraph tutorials could be: "LangGraph Tutorials in Practice"


## 5) Native function / tool calling (replaces `OPENAI_FUNCTIONS`)

`OPENAI_FUNCTIONS` opted an agent into OpenAI's function-calling schema instead of brittle text-based ReAct parsing. `create_agent` **always** uses the model's native tool-calling API when the model supports it — there's nothing extra to configure. Streaming the run lets us inspect the intermediate `AIMessage.tool_calls` to see the function-calling schema in action:

In [6]:
## 5) Native function / tool calling (replaces `OPENAI_FUNCTIONS`)
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Search the web for LangGraph docs and give me 3 bullets."}]},
    stream_mode="values",
):
    last = step["messages"][-1]
    if getattr(last, "tool_calls", None):
        print("Tool call(s):", [tc["name"] for tc in last.tool_calls])
    elif last.content:
        print("Final answer:\n", last.content)

Final answer:
 Search the web for LangGraph docs and give me 3 bullets.
Tool call(s): ['tavily_search']
Final answer:
 {"query": "LangGraph docs", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://docs.langchain.com/oss/python/langgraph/overview", "title": "LangGraph overview - Docs by LangChain", "content": "[Skip to main content](https://docs.langchain.com/oss/python/langgraph/overview#content-area). *   [Overview](https://docs.langchain.com/oss/python/langgraph/overview). *   [Install](https://docs.langchain.com/oss/python/langgraph/install). *   [Quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart). *   [Local server](https://docs.langchain.com/oss/python/langgraph/local-server). *   [Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph). *   [Workflows + agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents). *   [Persistence](https://docs.langchain.com/oss/python/la

## 6) Parallel tool calls (replaces `OPENAI_MULTI_FUNCTIONS`)

`OPENAI_MULTI_FUNCTIONS` let the agent call more than one tool within a single step. This is native behavior in v1 — when a prompt needs several tools at once, the model can emit multiple `tool_calls` on a single `AIMessage`, and `create_agent` executes them together, no special agent type needed.

In [7]:
## 6) Parallel tool calls (replaces `OPENAI_MULTI_FUNCTIONS`)
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use the simple_qa tool to explain what LangChain is in one sentence, "
                    "AND separately use web_search to find the latest LangChain release version."
                ),
            }
        ]
    }
)

for msg in result["messages"]:
    if getattr(msg, "tool_calls", None):
        print(f"{len(msg.tool_calls)} tool call(s) in this step:", [tc["name"] for tc in msg.tool_calls])

print("\nFinal answer:\n", result["messages"][-1].content)

2 tool call(s) in this step: ['simple_qa', 'tavily_search']

Final answer:
 LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively by providing tools to manage prompts, chain calls to language models, integrate external data sources, and handle memory and state.

The latest LangChain release version mentioned in the search results is 1.0.3.


## 7) Self-ask with search (replaces `SELF_ASK_WITH_SEARCH`)

The legacy self-ask agent forced a strict `Follow up: ... / Intermediate answer: ...` text format and required a search tool literally named `"Intermediate Answer"`. In v1 there's no dedicated agent type for this — instead we encode the same *reasoning pattern* in the `system_prompt` and let the model decide when to break a question into sub-questions before answering.

In [8]:
## 7) Self-ask with search (replaces `SELF_ASK_WITH_SEARCH`)
self_ask_prompt = """You are a careful research assistant that uses the self-ask technique:
1. Decide if the question needs to be broken into simpler follow-up questions.
2. For each follow-up question, use the web_search tool to find the answer.
3. Once you have every intermediate answer you need, combine them into one final answer.
Always ground factual claims in your search results."""

self_ask_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=self_ask_prompt,
)

result = self_ask_agent.invoke(
    {"messages": [{"role": "user", "content": "When was the first LangChain release and who founded it?"}]}
)
print(result["messages"][-1].content)

The first release of LangChain was in October 2022. LangChain was founded by Harrison Chase, who is the CEO and cofounder, along with co-founder Ankush Gola.


## Bonus: middleware — the real replacement for most `AgentType` behavior

In the legacy API, every new "flavor" of agent behavior (parsing-error recovery, memory summarization, human approval, etc.) needed its own `AgentType`. In v1, that customization surface is now **middleware**: composable hooks that wrap the model call, wrap each tool call, or run before/after an agent step.

A few prebuilt/custom middlewares that map to things the old notebook was doing manually or couldn't do at all:

- `handle_parsing_errors=True` (used on the old self-ask agent) → a `@wrap_tool_call` middleware that catches tool exceptions and returns a friendly `ToolMessage`
- Conversation memory getting too long → `SummarizationMiddleware`
- Requiring a human to approve a sensitive tool call → `HumanInTheLoopMiddleware`

See the [middleware guide](https://docs.langchain.com/oss/python/langchain/middleware/overview) for the full list of hooks (`before_agent`, `before_model`, `wrap_model_call`, `wrap_tool_call`, `after_model`, `after_agent`).

In [9]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

## Bonus: middleware — the real replacement for most `AgentType` behavior
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Equivalent to the old `handle_parsing_errors=True` flag."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: please rephrase your request. ({e})",
            tool_call_id=request.tool_call["id"],
        )


robust_agent = create_agent(
    model=model,
    tools=tools,
    middleware=[handle_tool_errors],
)

result = robust_agent.invoke(
    {"messages": [{"role": "user", "content": "Answer briefly from simple_qa; also check the web for any LangChain updates."}]}
)
print(result["messages"][-1].content)

LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively. It provides tools and components to manage prompts, chain multiple calls to language models, integrate with external data sources, and handle memory for complex AI applications.

Recent updates in LangChain include the release of LangChain v1, which is a focused, production-ready foundation for building agents. The framework has been streamlined around core improvements, with a new standard for building agents using the create_agent function. Legacy functionality has been moved to langchain-classic. Additionally, there have been various version updates and improvements in related libraries and tools. For detailed changes, you can refer to the LangChain v1 release notes and the GitHub releases page.


---
### Reference diagrams (carried over from the original notebook)

The image cells below (`agenttypes1.png`, `agenttype2.png`, `pro1.png`, `pro2.png`, `pro3.png`) illustrated the legacy `AgentType` architectures in the original notebook. The image files themselves were **not part of the uploaded notebook**, so they won't render unless you place the original PNGs in the same folder as this notebook. Conceptually, the v1 replacement for all of these diagrams is the single **`create_agent` + middleware** graph loop described at the top of this notebook — one graph, customized with middleware, instead of seven separate agent classes.

## Summary

- `initialize_agent` + `AgentType` → **deprecated**, moved to `langchain-classic`
- `create_agent` is now the single entry point for building agents in LangChain v1
- Behavior differences between the 7 legacy agent types are now expressed via **prompting choices**, **tool schemas**, and **middleware** — not via separate agent classes
- Memory is handled by **checkpointers** (`InMemorySaver`, `PostgresSaver`, ...) + `thread_id`, not `ConversationBufferMemory`
- Function calling and parallel tool calls are the **default** behavior, not a special mode

For more detail, see the official release notes: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>